# 07 · Validación multisemilla del detector YOLO11-s — SIRAP_2

**Objetivo:** completar seis entrenamientos independientes de YOLO11-s (base y trial 12 de Optuna, con semillas 0, 1 y 2) y medir la estabilidad de la diferencia de mAP@0.5:0.95 y recall.

Este notebook es independiente de `04_optimizacion.ipynb` y **NO** repite los 24 trials de Optuna. El notebook `06_afinamiento_ocr.ipynb` es para un estudio distinto de OCR y no se usa aquí.

**Requisitos:** Colab con GPU CUDA, acceso a Google Drive, secreto `ROBOFLOW_API_KEY`, y el archivo `MyDrive/proyecto_integrador_placas/optimizacion/optimizacion_resumen.json`.

**Seguridad de resultados:** los nuevos checkpoints se guardan solo en Drive, en `optimizacion/validacion_multisemilla`, sin modificar los pesos publicados en GitHub ni el Space de Hugging Face. Tres semillas permiten evaluar la variabilidad de forma descriptiva, no demostrar significación estadística.

Ejecuta **Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU** y después las celdas en orden.

## 1. Instalar las dependencias reproducibles

In [ ]:
%pip install -q "ultralytics==8.4.177" "roboflow==1.7.1" "pandas>=2.0" "PyYAML>=6.0"

## 2. Conectar Drive, comprobar CUDA y descargar el código más reciente

La sesión no puede iniciar entrenamientos hasta que esta comprobación detecte una GPU CUDA.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys
import torch
from google.colab import drive

drive.mount("/content/drive")
if not torch.cuda.is_available():
    raise RuntimeError("CUDA no está disponible. Selecciona Entorno de ejecución > Cambiar tipo de entorno de ejecución > T4 GPU.")
print("GPU disponible:", torch.cuda.get_device_name(0))

REPO = Path("/content/SIRAP_2")
if REPO.exists():
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "https://github.com/eduardoceballos666/SIRAP_2.git", str(REPO)], check=True)

WORK_DIR = Path("/content/drive/MyDrive/proyecto_integrador_placas")
OPT_SUMMARY = WORK_DIR / "optimizacion" / "optimizacion_resumen.json"
RESULTS_DIR = WORK_DIR / "optimizacion" / "validacion_multisemilla"
RUN_SCRIPT = REPO / "scripts" / "validar_multisemilla_yolo11.py"
assert OPT_SUMMARY.is_file(), f"No existe {OPT_SUMMARY}; revisa que Drive esté montado y el archivo disponible."
assert RUN_SCRIPT.is_file(), f"No existe {RUN_SCRIPT}; actualiza el repositorio en {REPO}."
print("Resumen Optuna:", OPT_SUMMARY)
print("Carpeta de resultados:", RESULTS_DIR)

## 3. Preparar Roboflow v11 con los splits originales

Se conserva el mismo conjunto de entrenamiento, validación y prueba que el estudio 04. La clave Roboflow se obtiene de los secretos de Colab y no se imprime.

In [ ]:
import yaml
from roboflow import Roboflow
from google.colab import userdata

RF_WORKSPACE = "roboflow-universe-projects"
RF_PROJECT = "license-plate-recognition-rxg4e"
RF_VERSION = 11
DATASET_DIR = Path("/content/datasets") / f"{RF_PROJECT}-v{RF_VERSION}"

found = sorted(DATASET_DIR.rglob("data.yaml"), key=lambda p: len(p.parts)) if DATASET_DIR.exists() else []
if not found:
    key = userdata.get("ROBOFLOW_API_KEY")
    if not key:
        raise RuntimeError("Agrega ROBOFLOW_API_KEY en los secretos de Colab antes de continuar.")
    Roboflow(api_key=key).workspace(RF_WORKSPACE).project(RF_PROJECT).version(RF_VERSION).download(
        "yolov11", location=str(DATASET_DIR)
    )
    found = sorted(DATASET_DIR.rglob("data.yaml"), key=lambda p: len(p.parts))
if not found:
    raise FileNotFoundError("La descarga Roboflow no creó data.yaml.")

root = found[0].parent
raw = yaml.safe_load(found[0].read_text())
cfg = {"path": str(root.resolve()), "names": raw["names"]}
for split, dirname in (("train", "train"), ("val", "valid"), ("test", "test")):
    folder = root / dirname / "images"
    if not folder.is_dir():
        raise FileNotFoundError(f"No existe el split {split}: {folder}")
    cfg[split] = str(folder.resolve())

DATA_YAML = root / "data_abs.yaml"
DATA_YAML.write_text(yaml.safe_dump(cfg, allow_unicode=True, sort_keys=False))
print("Dataset:", DATA_YAML)
print("Splits:", {key: cfg[key] for key in ("train", "val", "test")})

## 4. Ejecutar los seis entrenamientos completos

**Esta celda sí entrena modelos.** Usa 40 épocas, paciencia 10, AdamW y semillas 0, 1 y 2. Los resultados se guardan por corrida en Drive. No habilites ejecuciones simultáneas sobre la misma carpeta para evitar archivos incompatibles.

Si Colab pierde la conexión, puedes ejecutar el notebook de nuevo. El script reutiliza resultados ya evaluados y no sobrescribe automáticamente corridas incompletas. Si se interrumpe una corrida antes de terminar, revisa el checkpoint antes de intentar otra vez.

In [ ]:
command = [
    sys.executable, str(RUN_SCRIPT),
    "--data-yaml", str(DATA_YAML),
    "--opt-summary", str(OPT_SUMMARY),
    "--output-dir", str(RESULTS_DIR),
    "--seeds", "0", "1", "2",
    "--device", "0",
]
print("Iniciando la validación multisemilla en esta sesión CUDA:")
subprocess.run(command, check=True)

## 5. Mostrar resultados y variabilidad

**Solo después de terminar las seis corridas** deben usarse estas métricas en el informe. El test se utiliza para reportar la comparación fija, no para elegir nuevos hiperparámetros.

In [ ]:
import json
import pandas as pd
from IPython.display import display

csv_path = RESULTS_DIR / "comparacion_multisemilla.csv"
json_path = RESULTS_DIR / "resumen_multisemilla.json"
if not csv_path.exists() or not json_path.exists():
    raise FileNotFoundError("No se han generado los resultados: ejecuta primero la sección 4.")

resultados = pd.read_csv(csv_path)
resumen = json.loads(json_path.read_text())
display(resultados)
display(pd.DataFrame(resumen["comparaciones_pareadas"]))

print("Estado:", resumen["estado"])
print("Pares completos:", resumen["pares_completos"], "de 3")
print("Mejora media mAP@0.5:0.95:", resumen["delta_medio_map50_95"])
print("Desviación estándar de mejora:", resumen["delta_std_map50_95"])
print("Semillas con mejora:", resumen["semillas_con_mejora"], "de 3")
if resumen["estado"] != "completo":
    print("ATENCIÓN: resultados parciales; no publicar como validación final.")
else:
    print("Seis entrenamientos y tres pares registrados. Revisar cambios de recall y análisis de errores.")

## 6. Política de publicación

- Si la diferencia es pequeña frente a su variación o el recall disminuye, conservar el detector base.
- Incluso si la mejora es consistente, **no** publicar pesos optimizados directamente: primero verificar el rendimiento punta a punta, recalibrar el detector y volver a exportar ONNX.
- Documentar resultados reales en `docs/optimizacion.md` y `README.md`. No reutilizar el +0,0112 original como prueba multisemilla.
- Mantener las etiquetas SAVH como una tarea separada; el notebook OCR 06 no forma parte de estos seis entrenamientos.
